# Import the essentials

In [ ]:
# Imports here
from __future__ import print_function, division
import numpy as np
import matplotlib.pyplot as plt
from torch.utils import data
import torch
from torch import nn
from torch import optim
import torchvision
import torch.nn.functional as F
from torchvision import datasets, transforms, models
import torchvision.models as models
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from skimage import io, transform
import torch.utils.data as data_utils
from PIL import Image, ImageFile
from torch.optim import lr_scheduler
import os
import argparse
import seaborn as sns
import copy
import pandas as pd
ImageFile.LOAD_TRUNCATED_IMAGES = True
import cv2
# Import useful sklearn functions
import sklearn
from sklearn.metrics import cohen_kappa_score, accuracy_score
import time
from tqdm import tqdm_notebook
import seaborn as sns
import warnings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import seaborn as sns

# Loading Data + EDA

In [ ]:
train_csv = pd.read_csv('/content/drive/MyDrive/aptos2019-blindness-detection/train.csv')

test_csv = pd.read_csv('/content/drive/MyDrive/aptos2019-blindness-detection/test.csv')

In [ ]:
print('Train Size = {}'.format(len(train_csv)))
print('Public Test Size = {}'.format(len(test_csv)))

In [ ]:
train_csv.head()

In [ ]:
import matplotlib.pyplot as plt

import seaborn as sns



counts = train_csv['diagnosis'].value_counts()

class_list = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferate']



for i, x in enumerate(class_list):

    counts[x] = counts.pop(i)



plt.figure(figsize=(10, 5))

sns.barplot(x=counts.index, y=counts.values, alpha=0.8, palette='bright')

plt.title('Distribution of Output Classes')

plt.xlabel('Target Classes', fontsize=12)

plt.ylabel('Number of Occurrences', fontsize=12)

plt.show()

# Visualizing Training Data

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
# Define base directory

base_dir = '/content/drive/MyDrive/aptos2019-blindness-detection/'
# List of image files in the training images folder

train_imgs = os.listdir(os.path.join(base_dir, 'train_images'))



# Plotting images
fig = plt.figure(figsize=(30, 6))
# Display 16 random images
for idx, img in enumerate(np.random.choice(train_imgs, 16)):
    ax = fig.add_subplot(2, 16 // 2, idx + 1, xticks=[], yticks=[])
    im = Image.open(os.path.join(base_dir, "train_images", img))
    plt.imshow(im)
    lab = train_csv.loc[train_csv['id_code'] == img.split('.')[0], 'diagnosis'].values[0]
    ax.set_title('Severity: %s' % lab)

plt.show()

# Visualizing Test Data


In [ ]:
fig = plt.figure(figsize=(30, 6))

# display 20 images

test_imgs = os.listdir(base_dir+"/test_images")

for idx, img in enumerate(np.random.choice(test_imgs, 16)):

    ax = fig.add_subplot(2, 16//2, idx+1, xticks=[], yticks=[])

    im = Image.open(base_dir+"/test_images/" + img)

    plt.imshow(im)

# Data Pre Processing

In [ ]:
class CreateDataset(Dataset):

    def __init__(self, df_data, data_dir = '../input/', transform=None):

        super().__init__()

        self.df = df_data.values

        self.data_dir = data_dir

        self.transform = transform
        self.default_image = torch.zeros(3, 256, 256) # Define a default black image tensor


    def __len__(self):

        return len(self.df)



    def __getitem__(self, index):

        img_name,label = self.df[index]

        img_path = os.path.join(self.data_dir, img_name+'.png')
        # Check if file exists before reading
        if not os.path.exists(img_path):
            print(f"Warning: Image file not found at {img_path}. Returning placeholder.")
            return self.default_image, 0 # Return placeholder image and label 0


        try:
            image = Image.open(img_path).convert('RGB') # Ensure image is in RGB format
        except Exception as e:
            print(f"Warning: Could not read image file at {img_path} with PIL.Image.open: {e}. Returning placeholder.")
            return self.default_image, 0 # Return placeholder image and label 0


        if self.transform is not None:
            try:
                image = self.transform(image)
            except Exception as e:
                print(f"Warning: Error applying transform to image {img_name}.png: {e}. Returning placeholder.")
                return self.default_image, 0 # Return placeholder image and label 0


        return image, label

In [ ]:
train_transforms = transforms.Compose([

    # transforms.ToPILImage(), # This is redundant when using PIL.Image.open()

    transforms.Resize((256, 256)),

    transforms.RandomHorizontalFlip(p=0.4),

    transforms.ToTensor(),

    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

])

In [ ]:
test_transforms = transforms.Compose([transforms.Resize(256),

                                      transforms.CenterCrop(224),

                                      transforms.ToTensor(),

                                      transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])])

In [ ]:
train_path = "/content/drive/MyDrive/aptos2019-blindness-detection/train_images"

test_path = "/content/drive/MyDrive/aptos2019-blindness-detection/test_images"

In [ ]:
train_data = CreateDataset(df_data=train_csv, data_dir=train_path, transform=train_transforms)

test_data = CreateDataset(df_data=test_csv, data_dir=test_path, transform=test_transforms)

In [ ]:
valid_size = 0.2

num_train = len(train_data)

indices = list(range(num_train))

np.random.shuffle(indices)

split = int(np.floor(valid_size * num_train))

train_idx, valid_idx = indices[split:], indices[:split]

In [ ]:
train_sampler = SubsetRandomSampler(train_idx)

valid_sampler = SubsetRandomSampler(valid_idx)

In [ ]:
trainloader = torch.utils.data.DataLoader(train_data, batch_size=64,sampler=train_sampler)

validloader = torch.utils.data.DataLoader(train_data, batch_size=64, sampler=valid_sampler)

testloader = torch.utils.data.DataLoader(test_data, batch_size=64)

In [ ]:
print(f"training examples contain : {len(train_data)}")

print(f"testing examples contain : {len(test_data)}")

In [ ]:
images, labels = next(iter(trainloader))

# Checking shape of image
print(f"Type of images after loading: {type(images)}") # Debug print
print(f"Shape of images after loading: {images.shape}") # Debug print
print(f"Type of labels after loading: {type(labels)}") # Debug print
print(f"Shape of labels after loading: {labels.shape}") # Debug print


# denormalizing images for printing as they are in tensors

def imshow(inp, title=None):

    """Imshow for Tensor."""

    # If the image is None, display a blank image or skip
    if inp is None:
        print("Warning: Skipping display of a None image.")
        return

    # Move the tensor to CPU before converting to numpy
    inp = inp.cpu()
    inp = inp.numpy().transpose((1, 2, 0))

    mean = np.array([0.485, 0.456, 0.406])

    std = np.array([0.229, 0.224, 0.225])

    inp = std * inp + mean

    inp = np.clip(inp, 0, 1)

    plt.imshow(inp)

    if title is not None:

        plt.title(title)

    plt.pause(0.001)

print(f"Type of 'out' before calling imshow: {type(images)}") # Debug print - assuming 'out' is the images tensor here

In [ ]:
# plotting the images of loaded batch with given fig size and frame data

import torchvision

import matplotlib.pyplot as plt

import numpy as np

grid = torchvision.utils.make_grid(images, nrow=30, padding=7)

# Move the tensor to CPU before transposing

grid = grid.cpu()

plt.figure(figsize=(20, 20))

plt.imshow(np.transpose(grid, (1, 2, 0)))

print('labels:', labels)

In [ ]:
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']



images, labels = next(iter(trainloader))

# Filter out None images and their corresponding labels
valid_images = [images[i] for i, img in enumerate(images) if img is not None]
valid_labels = [labels[i] for i, img in enumerate(images) if img is not None]


if valid_images:
    out = torchvision.utils.make_grid(valid_images)
    print(f"Type of 'out' after make_grid: {type(out)}") # Debug print
    print(f"Content of 'out' after make_grid: {out}") # Debug print


    imshow(out, title=[class_names[x] for x in valid_labels])
else:
    print("No valid images to display in the batch.")

In [ ]:
###SHUTIL####



import os

import shutil

import random



# Set the directory paths

train = '/content/drive/MyDrive/aptos2019-blindness-detection/train_images'

test = '/content/drive/MyDrive/aptos2019-blindness-detection/test_images'

# Create the validation directory if it doesn't exist

os.makedirs('test' , exist_ok=True)

os.makedirs('train' , exist_ok=True)

os.makedirs('val' , exist_ok=True)



t_df = pd.read_csv('/content/drive/MyDrive/aptos2019-blindness-detection/train.csv')



train_df = t_df.sample(frac = 0.80)

test_df = t_df.drop(train_df.index)

test_df.reset_index(drop=True, inplace=True)



val_df = test_df.sample(frac = 0.45)

test_df = test_df.drop(val_df.index)

test_df.reset_index(drop=True, inplace=True)

val_df.reset_index(drop=True, inplace=True)



train_df.sort_values(by='diagnosis', inplace=True)

train_df.reset_index(drop=True, inplace=True)



test_df.sort_values(by='diagnosis', inplace=True)

test_df.reset_index(drop=True, inplace=True)



val_df.sort_values(by='diagnosis', inplace=True)

val_df.reset_index(drop=True, inplace=True)

In [ ]:
train_df['diagnosis'].value_counts().plot(kind='pie' , labels = list(train_df['diagnosis'].value_counts()))

In [ ]:
test_df['diagnosis'].value_counts().plot(kind='pie' , labels = list(test_df['diagnosis'].value_counts()))

In [ ]:
val_df['diagnosis'].value_counts().plot(kind='pie' , labels = list(val_df['diagnosis'].value_counts()))

In [ ]:
for i in range(5):

  imgs = train_df[train_df['diagnosis'] == i]['id_code'].to_list()

  os.makedirs(os.path.join('train', str(i)), exist_ok=True)

  for j in imgs:

    shutil.copy(os.path.join(train, j+'.png'), os.path.join(f'train/{i}', str(j)+'.png'))

In [ ]:
for i in range(5):

  imgs = test_df[test_df['diagnosis'] == i]['id_code'].to_list()

  os.makedirs(os.path.join('test', str(i)), exist_ok=True)

  for j in imgs:

    shutil.copy(os.path.join(train, j+'.png'), os.path.join(f'test/{i}', str(j)+'.png'))

In [ ]:
for i in range(5):

  imgs = val_df[val_df['diagnosis'] == i]['id_code'].to_list()

  os.makedirs(os.path.join('val', str(i)), exist_ok=True)

  for j in imgs:

    shutil.copy(os.path.join(train, j+'.png'), os.path.join(f'val/{i}', str(j)+'.png'))

In [ ]:
import os
directory_path = "./test" # Corrected path
# Get the list of directories inside the specified directory
elements = os.listdir(directory_path)
print(f"Number of elements in each directory inside '{directory_path}':")
for element in elements:
    full_path = os.path.join(directory_path, element)
    if os.path.isdir(full_path):  # Check if the element is a directory
        num_elements = len(os.listdir(full_path))
        print(f"Directory: {element}, Number of elements: {num_elements}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score

# Define number of classes
num_classes = 5

# Define placeholder paths for your model weight files in Google Drive
# !!! IMPORTANT: Replace these with the actual paths to your saved model weights in Google Drive !!!
resnet50_weights_path = '/content/drive/MyDrive/DR_Weights/resnet50_best_model.pth' # REPLACE WITH ACTUAL PATH
densenet201_weights_path = '/content/drive/MyDrive/DR_Weights/densenet201_best.pth' # REPLACE WITH ACTUAL PATH
densenet121_weights_path = '/content/drive/MyDrive/DR_Weights/best_modeldensenet121.pth' # REPLACE WITH ACTUAL PATH


# Load and modify ResNet50
resnet50 = models.resnet50(pretrained=False)
resnet50.fc = nn.Linear(resnet50.fc.in_features, num_classes)
# Update the path to load state_dict from Google Drive
state_dict = torch.load(resnet50_weights_path, map_location=torch.device('cpu'))
# Note: The keys in your saved state_dict might be different if it was saved from a model with different layer names.
# The original code assumes 'last_linear.weight' and 'last_linear.bias'. If your keys are different, you'll need to adjust the pop() calls.
if 'last_linear.weight' in state_dict and 'last_linear.bias' in state_dict:
    state_dict['fc.weight'] = state_dict.pop('last_linear.weight')
    state_dict['fc.bias'] = state_dict.pop('last_linear.bias')
elif 'fc.weight' in state_dict and 'fc.bias' in state_dict:
     pass # Keys are already in the expected format
else:
     print("Warning: Could not find expected keys ('last_linear.weight'/'last_linear.bias' or 'fc.weight'/'fc.bias') in ResNet50 state_dict. Loading might fail.")

resnet50.load_state_dict(state_dict)
resnet50.eval()

# Load and modify DenseNet201
densenet201 = models.densenet201(pretrained=False)
densenet201.classifier = nn.Linear(densenet201.classifier.in_features, num_classes)
# Update the path to load state_dict from Google Drive
densenet201.load_state_dict(torch.load(densenet201_weights_path, map_location=torch.device('cpu')))
densenet201.eval()

# Load and modify DenseNet121 (instead of MobileNetV2)
densenet121 = models.densenet121(pretrained=False)
# Modify classifier layer to handle the mismatch
densenet121.classifier = nn.Linear(densenet121.classifier.in_features, num_classes)

# Load the model state dict
# Update the path to load state_dict from Google Drive
state_dict_densenet121 = torch.load(densenet121_weights_path, map_location=torch.device('cpu'))

# Since the classifier layers have different names, we'll rename them in the state_dict
# Rename 'classifier.1.weight' and 'classifier.1.bias' to 'classifier.weight' and 'classifier.bias'
# Add checks to see if the keys exist before trying to pop/rename
if 'classifier.1.weight' in state_dict_densenet121 and 'classifier.1.bias' in state_dict_densenet121:
    state_dict_densenet121['classifier.weight'] = state_dict_densenet121.pop('classifier.1.weight')
    state_dict_densenet121['classifier.bias'] = state_dict_densenet121.pop('classifier.1.bias')
elif 'classifier.weight' in state_dict_densenet121 and 'classifier.bias' in state_dict_densenet121:
     pass # Keys are already in the expected format
else:
     print("Warning: Could not find expected keys ('classifier.1.weight'/'classifier.1.bias' or 'classifier.weight'/'classifier.bias') in DenseNet121 state_dict. Loading might fail.")


# Load the adjusted state dict into the model
densenet121.load_state_dict(state_dict_densenet121)
densenet121.eval()

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet50.to(device)
densenet201.to(device)
densenet121.to(device)

# Define the data transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Ensure compatible input size for all models
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load the test dataset
# Update test_dir to point to the directory containing the test images organized by class
test_dir = './test' # Assuming './test' is the correct path to the test data organized into class folders
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Evaluation function
def evaluate_model(model, test_loader, device):
    all_preds = []
    all_labels = []
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Get model predictions
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            # Count correct predictions
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return all_preds, all_labels, accuracy

# Perform evaluation for each model and collect predictions for ensemble
all_preds_resnet50 = []
all_preds_densenet201 = []
all_preds_densenet121 = []
all_labels = []

# Get predictions for all models
# Iterate through the test_loader to get actual labels
actual_labels_list = []
for images, labels in test_loader:
     actual_labels_list.extend(labels.cpu().numpy())


# Re-iterate through the test_loader to get predictions from each model
# This is necessary because the previous loop consumed the iterator
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False) # Re-create the test_loader

for images, labels in test_loader: # Use labels from the second iteration for prediction collection
    images = images.to(device)
    # No need to move labels to device here as they are not used for prediction

    # ResNet50 predictions
    outputs_resnet50 = resnet50(images)
    _, predicted_resnet50 = torch.max(outputs_resnet50, 1)

    # DenseNet201 predictions
    outputs_densenet201 = densenet201(images)
    _, predicted_densenet201 = torch.max(outputs_densenet201, 1)

    # DenseNet121 predictions
    outputs_densenet121 = densenet121(images)
    _, predicted_densenet121 = torch.max(outputs_densenet121, 1)

    # Collect predictions
    all_preds_resnet50.extend(predicted_resnet50.cpu().numpy())
    all_preds_densenet201.extend(predicted_densenet201.cpu().numpy())
    all_preds_densenet121.extend(predicted_densenet121.cpu().numpy())


# Ensemble prediction (majority voting)
ensemble_preds = []
for i in range(len(actual_labels_list)):
    # Get the predictions for the current image from all models
    model_preds = [all_preds_resnet50[i], all_preds_densenet201[i], all_preds_densenet121[i]]

    # Majority vote for ensemble prediction
    ensemble_pred = np.bincount(model_preds).argmax()  # Most common prediction
    ensemble_preds.append(ensemble_pred)

# Calculate evaluation metrics for the ensemble model
# Use actual_labels_list for the true labels

# First, calculate ensemble accuracy (needed for print statement)
ensemble_correct = sum(1 for i in range(len(actual_labels_list)) if actual_labels_list[i] == ensemble_preds[i])
ensemble_accuracy = ensemble_correct / len(actual_labels_list)

precision = precision_score(actual_labels_list, ensemble_preds, average='weighted')
recall = recall_score(actual_labels_list, ensemble_preds, average='weighted')
f1 = f1_score(actual_labels_list, ensemble_preds, average='weighted')
kappa_score = cohen_kappa_score(actual_labels_list, ensemble_preds)

# Print ensemble results
print("Ensemble Model Results:")
print(f"Ensemble Test Accuracy: {ensemble_accuracy * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Cohen's Kappa Score: {kappa_score:.4f}")


In [ ]:
import os

# Define the path to your Google Drive mount point
drive_path = '/content/drive/MyDrive'

# Check if the directory exists
if os.path.exists(drive_path):
    print(f"Google Drive is mounted at {drive_path}.")
    # List the contents of the Google Drive
    print("Contents of Google Drive:")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"Google Drive is not mounted at {drive_path}. Please run the cell to mount Google Drive.")